### Importing Libraries

In [1]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

### Regular Run

In [2]:
lenet_transforms = transforms.Compose([
    transforms.Resize((28, 28)),    
    transforms.ToTensor(),        
    transforms.Normalize((0.1307,), (0.3081,))
])

In [3]:
train_dataset = datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=lenet_transforms
)

test_dataset = datasets.MNIST(
    root='./data', 
    train=False, 
    transform=lenet_transforms
)

100%|██████████| 9.91M/9.91M [00:03<00:00, 3.11MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 120kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 901kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.23MB/s]


In [4]:
train_loader = DataLoader(
    train_dataset,
    batch_size=5,
    shuffle=True,
)
test_loader = DataLoader(test_dataset, batch_size=5, shuffle=True)

#### Custom LeCun "SigmoidLike" Activation

$$
\begin{aligned}
x_{i} &= f(a_{i})\\
f(a) &= A \tanh (Sa)
\end{aligned}
$$

In [9]:
class SigmoidLike(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.A = 1.17159
        self.S = 2/3
        
    def forward(self, x):
        return self.A  * torch.tanh(self.S * x)

## Model

In [7]:
class LeNet5(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.conv1 = nn.Conv2d(in_channels=1,out_channels=6,kernel_size=5)
        self.avgpool1 = nn.AvgPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(in_channels=6,out_channels=16,kernel_size=5)
        self.avgpool2 = nn.AvgPool2d(kernel_size=2)
        self.conv3 = nn.Conv2d(in_channels=16,out_channels=120,kernel_size=4)
        self.fc1 = nn.Linear(in_features=120,out_features=84)

        self.act = SigmoidLike()


    
    def forward(self, x):
        x = self.conv1(x)
        x = self.act(x)

        x = self.avgpool1(x)
        x = self.act(x)

        x = self.conv2(x)
        x = self.act(x)

        x = self.avgpool2(x)
        x = self.act(x)

        x = self.conv3(x) 
        x = self.act(x)
        x = x.view(x.size(0), -1)

        x = self.fc1(x)

        return x

### Model Init

In [10]:
model = LeNet5()
model.to(device="cuda")

LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (avgpool1): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (avgpool2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv3): Conv2d(16, 120, kernel_size=(4, 4), stride=(1, 1))
  (fc1): Linear(in_features=120, out_features=84, bias=True)
  (act): SigmoidLike()
)